# Extract clean facts with Llama 70B

Start from `runs/MuSiQue_final_questions.pkl`. Test a single paragraph first, then run all questions with the same loaded model. New facts contain only `text`, `paragraph_idx`, `paragraph_title`, and `is_supporting`. No fact judging, triple generation, or ONA execution is performed.

Use a GPU with sufficient memory for the roughly 40 GB model plus inference memory. Ensure the updated local `facts.py` and `filter.py` are committed to GitHub before cloning, or upload them into the cloned directory after setup. Never run the old `numpy<2` downgrade cell.

Setup installs packages following [GPTQModel installation](https://github.com/ModelCloud/GPTQModel#install-). Restart the session after setup, then execute Cells 1–3. Inspect the preview before executing Cell 4. Cell 5 publishes code and artifacts when you run it.


## Setup — clone and install


In [ ]:
# Run once in a fresh Colab GPU session.
from pathlib import Path
import os
import subprocess

REPO = Path("/content/llm-vs-reasoning")
if not (REPO / ".git").exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/MinaGabriel/llm-vs-reasoning.git",
        str(REPO),
    ], check=True)

os.chdir(REPO)

%pip install -q -U "numpy>=2.1,<3" transformers accelerate prettytable bitsandbytes gptqmodel ninja

print("Setup complete. Restart the Colab session once, then run Cell 1.")


## Cell 1 — imports and questions


In [ ]:
# Cell 1: Imports and input questions.
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import copy
import importlib
import inspect
import json
import pickle
import shutil
import subprocess
import tempfile
import time
from pathlib import Path

import numpy as np
import torch
from prettytable import PrettyTable
from google.colab import userdata

REPO = Path("/content/llm-vs-reasoning")
os.chdir(REPO)

import facts as FA
import filter as F
importlib.reload(F)
importlib.reload(FA)

# Stop here if GitHub still has the older facts.py.
if not hasattr(FA, "extract_facts") or list(
    inspect.signature(FA.extract_facts).parameters
) != ["llm", "paragraph_lists", "batch_size"]:
    raise RuntimeError(
        "Upload the latest local facts.py and filter.py into "
        "/content/llm-vs-reasoning, then rerun this cell."
    )

INPUT = Path("runs/MuSiQue_final_questions.pkl")
OUTPUT = Path("runs/MuSiQue_final_questions_facts_clean.pkl")
MODEL_ID = "hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4"
BATCH_SIZE = 4
FACT_KEYS = {"text", "paragraph_idx", "paragraph_title", "is_supporting"}

with INPUT.open("rb") as handle:
    questions = pickle.load(handle)
if not questions:
    raise ValueError("The questions file is empty.")

print("NumPy:", np.__version__)
print("Questions:", len(questions))
print("First question:", questions[0]["question"])
print("Output:", OUTPUT)


## Cell 2 — load Llama once


In [ ]:
# Cell 2: Load Llama once. Keep it loaded for Cells 3 and 4.
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before loading this model.")

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; total memory: {gpu.total_memory / 2**30:.1f} GiB")

if "llm" not in globals() or llm is None:
    llm = F.load_llm(MODEL_ID)
else:
    print("Reusing the loaded model.")

print("Llama is ready.")


## Cell 3 — single paragraph preview

Fact extraction takes a paragraph as input and returns multiple fact sentences. This cell tests one paragraph from the first question.


In [ ]:
# Cell 3: Test one original paragraph and print all facts extracted from it.
QUESTION_INDEX = 0
source = questions[QUESTION_INDEX]

# Choose the first supporting paragraph. Change this index to test another.
PARAGRAPH_INDEX = next(
    (i for i, p in enumerate(source["paragraphs"]) if p.get("is_supporting")),
    0,
)
paragraph = source["paragraphs"][PARAGRAPH_INDEX]

started = time.time()
sample_facts, n_unique = FA.extract_facts(
    llm, [[paragraph]], batch_size=1,
)

# Restore the source paragraph's position in the original question.
for fact in sample_facts[0]:
    fact["paragraph_idx"] = PARAGRAPH_INDEX

sample = copy.deepcopy(source)
FA.attach_facts(
    [sample], sample_facts, n_unique, MODEL_ID, time.time() - started,
)

if not sample["facts"]:
    raise RuntimeError("No facts were parsed. Inspect the extraction before continuing.")
assert all(set(fact) == FACT_KEYS for fact in sample["facts"]), (
    "Unexpected fact fields: upload the latest facts.py and rerun Cell 1."
)

print("Question:", sample["question"])
print("Paragraph:", paragraph["title"])

table = PrettyTable()
table.field_names = ["Original paragraph", "Extracted facts"]
table.align = "l"
table.valign = "t"
table.max_width = 65
table.add_row([
    paragraph["text"],
    "\n\n".join(
        f"{i}. {fact['text']}"
        for i, fact in enumerate(sample["facts"], 1)
    ),
])
print(table)
print("\nActual fact structure:")
print(json.dumps(sample["facts"], indent=2, ensure_ascii=False))

preview_path = Path("runs/facts_preview.json")
preview_path.parent.mkdir(parents=True, exist_ok=True)
preview_path.write_text(
    json.dumps({
        "question": sample["question"],
        "paragraph": paragraph,
        "facts": sample["facts"],
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Preview saved:", preview_path)


## Cell 4 — full facts generation

This runs only the final fact-extraction stage, starting from the selected questions. It does not repeat dataset filtering or run NARS. It updates `MuSiQue_final.pkl` and saves a separate clean facts pickle.


In [ ]:
# Cell 4: Run this only after you are satisfied with Cell 3.
# Processes ALL questions and ALL paragraphs, including distractors.
records = copy.deepcopy(questions)
started = time.time()

all_facts, n_unique = FA.extract_facts(
    llm,
    [record.get("paragraphs") for record in records],
    batch_size=BATCH_SIZE,
)
entry = FA.attach_facts(
    records, all_facts, n_unique, MODEL_ID, time.time() - started,
    {"input_file": str(INPUT), "output_file": str(OUTPUT)},
)

assert all(
    set(fact) == FACT_KEYS
    for record in records
    for fact in record["facts"]
), "Unexpected fact schema; the output has not been saved."
empty_questions = [record["id"] for record in records if not record["facts"]]
if empty_questions:
    raise RuntimeError(f"No facts extracted for questions: {empty_questions}")

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT.open("wb") as handle:
    pickle.dump(records, handle)

# Update the project's final dataset with this clean facts version.
shutil.copy2(OUTPUT, "MuSiQue_final.pkl")
F.save_stats(records[0]["dataset"], entry)

print("Questions:", len(records))
print("Unique paragraphs:", n_unique)
print("Facts:", entry["facts_kept"])
print("Elapsed seconds:", entry["elapsed_sec"])
print("Saved:", OUTPUT)
print("Updated: MuSiQue_final.pkl")
print(json.dumps(records[0]["facts"][:3], indent=2, ensure_ascii=False))


## Cell 5 — commit and push

Provide `GITHUB_TOKEN` through Colab Secrets. This cell commits tracked changes and project output artifacts, then pushes to the current branch. It does not force-push or store the token in Git configuration.


In [ ]:
# Cell 5: Commit and push code, stats, notebooks, and generated data.
# Add GITHUB_TOKEN in Colab's Secrets panel and grant this notebook access.
# Running this cell publishes the staged changes to the current GitHub branch.
os.chdir(REPO)
if not OUTPUT.exists():
    raise RuntimeError("Run Cell 4 successfully before pushing.")

REMOTE = "https://github.com/MinaGabriel/llm-vs-reasoning.git"
subprocess.run(["git", "remote", "set-url", "origin", REMOTE], check=True)
subprocess.run(["git", "config", "user.name", "Mina Gabriel"], check=True)

email = subprocess.run(
    ["git", "config", "--get", "user.email"],
    capture_output=True, text=True,
).stdout.strip()
if not email or email == "mina@example.com":
    email = input("Git commit email (GitHub verified or noreply address): ").strip()
    if not email:
        raise ValueError("A Git commit email is required.")
    subprocess.run(["git", "config", "user.email", email], check=True)

branch = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True,
).strip()
if not branch:
    raise RuntimeError("Checkout a Git branch before pushing.")

# Include tracked changes, plus project code, notebooks, and output artifacts.
# Keep downloaded models, caches, and the separate ONA source checkout out of Git.
subprocess.run(["git", "add", "-u"], check=True)
candidates = {
    path
    for pattern in ("*.py", "*.json", "*.pkl", "*.ipynb")
    for path in Path(".").glob(pattern)
    if path.is_file()
}
candidates.update(
    path for path in Path("runs").rglob("*")
    if path.is_file() and path.suffix in {".pkl", ".json", ".nal", ".txt"}
)
paths = [
    str(path) for path in sorted(candidates)
    if subprocess.run(
        ["git", "check-ignore", "-q", "--", str(path)]
    ).returncode != 0
]
if paths:
    subprocess.run(["git", "add", "--", *paths], check=True)

subprocess.run(["git", "diff", "--cached", "--stat"], check=True)
changed = subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode
if changed == 1:
    subprocess.run([
        "git", "commit", "-m",
        "Generate clean facts without probability judging or per-fact model metadata",
    ], check=True)
elif changed != 0:
    raise RuntimeError("Could not inspect staged changes.")
else:
    print("No new changes to commit; pushing any existing local commits.")

# Authenticate without placing the token in the remote URL or notebook output.
with tempfile.TemporaryDirectory() as directory:
    askpass = Path(directory) / "git-askpass.sh"
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$COLAB_GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["COLAB_GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    try:
        subprocess.run(
            ["git", "-c", "credential.helper=", "push", "origin", f"HEAD:{branch}"],
            env=env,
            check=True,
        )
    finally:
        env.pop("COLAB_GITHUB_TOKEN", None)

print("Pushed to branch:", branch)
